# ISL Bridge: model training on Google Colab
**Before you run:** Runtime > Change runtime type > **GPU**.

Expected Drive layout:
```
MyDrive/
  ISL_Bridge.zip        <- the website
  INCLUDE/              <- the INCLUDE dataset (zip file(s), or already extracted videos)
```
What this notebook does, in order:
1. unpacks the website and the dataset
2. extracts hand keypoints from every video (progress is saved to Drive, so it can resume if Colab disconnects)
3. trains the BiLSTM + Transformer model, then runs a federated-averaging round simulation
4. writes `isl_model.pt`, `labels.json`, `pose_library.json`, `replay.pt` straight into the website's `model/` folder
5. saves the finished website as `MyDrive/ISL_Bridge_trained.zip`

In [ ]:
!pip -q install mediapipe opencv-python-headless scikit-learn tqdm

## 1. Drive, website and dataset

### 1a. Download the INCLUDE word videos (Greetings + Adjectives) straight into Colab
This cell downloads only `Greetings_1of2/2of2` and `Adjectives_1of8..4of8` from the official INCLUDE dataset on Zenodo (about 8 GB). They go to Colab's temporary disk, **nothing needs to be uploaded to Google Drive**. Each file is checked against its published MD5 checksum. If Colab disconnects, run this cell again: finished files are skipped and half-finished ones resume.

In [ ]:
import os, subprocess, hashlib, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

DL = "/content/include_zips"; os.makedirs(DL, exist_ok=True)
BASE = "https://zenodo.org/records/4010759/files/"
# file name -> MD5 published on https://zenodo.org/records/4010759  (INCLUDE, CC-BY 4.0)
FILES = {
    "Greetings_1of2.zip":  "457624ce89b2d32520d6b61bd1f113a2",
    "Greetings_2of2.zip":  "2dd843662545b0ca57e79cff84562698",
    "Adjectives_1of8.zip": "aaa8281b31454aab89ea73ac2536d87f",
    "Adjectives_2of8.zip": "3ab2ac5807fe618db236b1986f61518e",
    "Adjectives_3of8.zip": "a39b68944aa8d0b64d794eb67c95d523",
    "Adjectives_4of8.zip": "62e20ec505ff28424c57cc69b26828e4",
}
print("free disk: %.0f GB (about 17 GB needed: zips + unpacked videos)" % (shutil.disk_usage("/content").free / 1e9))

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

def fetch(item):
    name, md5 = item; out = f"{DL}/{name}"
    for attempt in range(3):
        if os.path.exists(out) and md5sum(out) == md5: return name, True
        subprocess.run(["wget", "-c", "-q", "--tries=3", "--timeout=60", "-O", out, f"{BASE}{name}?download=1"])
        if os.path.exists(out) and md5sum(out) == md5: return name, True
        if os.path.exists(out): os.remove(out)           # corrupt or incomplete: start that file again
    return name, False

failed = []
with ThreadPoolExecutor(max_workers=3) as ex:
    for fut in as_completed([ex.submit(fetch, it) for it in FILES.items()]):
        name, ok = fut.result()
        print(("OK      " if ok else "FAILED  ") + name, "%.2f GB" % (os.path.getsize(f"{DL}/{name}") / 1e9) if ok else "")
        if not ok: failed.append(name)
if failed:
    raise RuntimeError("These files did not download correctly: %s. Run this cell again (it resumes), "
                       "or check that Colab has internet access and zenodo.org is reachable." % failed)
print("all", len(FILES), "files downloaded and verified")


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)   # remount so newly added files show up
import os, glob, zipfile, sys, shutil

DRIVE = "/content/drive/MyDrive"
top = sorted(os.listdir(DRIVE))
print(len(top), "items in My Drive. First ones:")
for n in top[:60]: print("  ", repr(n), "(folder)" if os.path.isdir(f"{DRIVE}/{n}") else "")

def find(name, want_dir, depth=3):
    """Case-insensitive search (ignores stray spaces) for a folder or file in My Drive."""
    name = name.lower()
    for root, dirs, fs in os.walk(DRIVE):
        if root[len(DRIVE):].count(os.sep) >= depth: dirs[:] = []
        for n in (dirs if want_dir else fs):
            if n.strip().lower() == name:
                return os.path.join(root, n)
    return None

WEB_ZIP  = find("ISL_Bridge.zip", False)
DATA_DIR = find("INCLUDE", True)
WORK     = f"{DRIVE}/ISL_Bridge_work"          # progress is saved here
os.makedirs(WORK, exist_ok=True)

if not WEB_ZIP:
    raise FileNotFoundError("ISL_Bridge.zip not found in My Drive. Check the file name in the list above.")
if not DATA_DIR:
    raise FileNotFoundError(
        "No real folder named INCLUDE is visible to Colab. If INCLUDE is a shortcut or lives in 'Shared with me', "
        "Colab cannot read it: right-click it in Drive > Make a copy (or move the zip files into a normal folder "
        "inside My Drive), then run this cell again.")
print("website zip :", WEB_ZIP)
print("dataset dir :", DATA_DIR, "->", [os.path.basename(p) for p in glob.glob(DATA_DIR + "/*")][:10])

# --- website ---
SITE_TMP = "/content/site"
shutil.rmtree(SITE_TMP, ignore_errors=True)
zf = zipfile.ZipFile(WEB_ZIP); zf.extractall(SITE_TMP)
names = zf.namelist()
print(len(names), "entries in", os.path.basename(WEB_ZIP), "- first ones:")
for n in names[:25]: print("  ", n)

ISL_MODEL_PY = r'''import torch
import torch.nn as nn

SEQ_LEN = 30      # frames per sign
FEATS = 126       # 2 hands x 21 points x (x, y, z)


class ISLNet(nn.Module):
    """BiLSTM front end -> Transformer encoder -> attention pooling -> classifier."""

    def __init__(self, n_classes, hidden=128, heads=4, layers=2, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(FEATS, hidden, batch_first=True, bidirectional=True)
        d = hidden * 2
        self.pos = nn.Parameter(torch.zeros(1, SEQ_LEN, d))
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, drop, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.score = nn.Linear(d, 1)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, n_classes))

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.encoder(h + self.pos[:, : h.size(1)])
        w = torch.softmax(self.score(h), dim=1)
        return self.head((w * h).sum(1))
'''
PREDICTOR_PY = r'''import json, os
import numpy as np
import torch
from .isl_model import ISLNet, SEQ_LEN, FEATS

# words (labels longer than one character) are favoured over letters/digits; 1 = off
WORD_BOOST = float(os.environ.get("WORD_BOOST", "3"))


class Predictor:
    def __init__(self, folder):
        w, l = os.path.join(folder, "isl_model.pt"), os.path.join(folder, "labels.json")
        self.ready = os.path.exists(w) and os.path.exists(l)
        if not self.ready:
            return
        with open(l, encoding="utf8") as f:
            self.labels = json.load(f)
        self.boost = torch.tensor([WORD_BOOST if len(x) > 1 else 1.0 for x in self.labels])
        self.net = ISLNet(len(self.labels))
        self.net.load_state_dict(torch.load(w, map_location="cpu"))
        self.net.eval()

    @torch.no_grad()
    def predict(self, seq, k=3):
        a = np.asarray(seq, dtype=np.float32)
        if a.ndim != 2 or a.shape[1] != FEATS or len(a) < 4:
            raise ValueError("bad sequence shape")
        a = a[np.linspace(0, len(a) - 1, SEQ_LEN).round().astype(int)]
        p = torch.softmax(self.net(torch.from_numpy(a)[None]), 1)[0] * self.boost
        p = p / p.sum()
        v, i = p.topk(min(k, len(self.labels)))
        return [{"label": self.labels[j], "conf": round(float(c), 3)} for c, j in zip(v, i)]
'''

marker = glob.glob(SITE_TMP + "/**/model/isl_model.py", recursive=True)
if marker:
    MODEL_DIR = os.path.dirname(marker[0]); SITE = os.path.dirname(MODEL_DIR)
else:
    # the zip does not contain the model code: use the built-in copy and put it next to app.py (if there is one)
    app = glob.glob(SITE_TMP + "/**/app.py", recursive=True)
    SITE = os.path.dirname(app[0]) if app else SITE_TMP + "/ISL_Bridge"
    MODEL_DIR = SITE + "/model"; os.makedirs(MODEL_DIR, exist_ok=True)
    for fn, code in (("isl_model.py", ISL_MODEL_PY), ("predictor.py", PREDICTOR_PY), ("__init__.py", "")):
        if not os.path.exists(f"{MODEL_DIR}/{fn}"):
            open(f"{MODEL_DIR}/{fn}", "w").write(code)
    print("isl_model.py was not in the zip, so the built-in copy was written to", MODEL_DIR)
    if not app:
        print("NOTE: no app.py found in the zip either. The output zip will hold only the model/ files,")
        print("      copy them into your website's model/ folder.")
sys.path.insert(0, SITE)
print("website folder:", SITE)
print("model folder  :", MODEL_DIR)

# --- dataset: unpack every archive in INCLUDE, including zips nested inside zips ---
import tarfile, collections
EXTRACT = "/content/include_data"
os.makedirs(EXTRACT, exist_ok=True)
ARCH = (".zip", ".tar", ".tar.gz", ".tgz", ".tar.bz2")
free = lambda: "%.0f GB free" % (shutil.disk_usage("/content").free / 1e9)
print("disk:", free())

def unpack(src, out):
    """Extract one archive into `out`. A .done marker means it finished, so a half-finished
    extraction from an earlier run is redone instead of being trusted."""
    if os.path.exists(out + "/.done"): return True
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    try:
        if src.lower().endswith(".zip"): zipfile.ZipFile(src).extractall(out)
        else: tarfile.open(src).extractall(out)
        open(out + "/.done", "w").close(); return True
    except Exception as e:
        print("  !! failed:", os.path.basename(src), "->", type(e).__name__, e); return False

top_arch = sorted(p for p in glob.glob(DATA_DIR + "/**/*", recursive=True) if p.lower().endswith(ARCH))
top_arch += sorted(glob.glob("/content/include_zips/*.zip"))      # parts downloaded by cell 1a
other = sorted(p for p in glob.glob(DATA_DIR + "/**/*", recursive=True) if p.lower().endswith((".rar", ".7z")))
print(len(top_arch), "archive(s) in INCLUDE:", [os.path.basename(p) for p in top_arch])
if other: print("!! .rar/.7z cannot be opened here, re-upload as .zip:", [os.path.basename(p) for p in other])
for z in top_arch:
    print("unpacking", os.path.basename(z), "(%.1f GB)" % (os.path.getsize(z) / 1e9), "- this can take several minutes")
    unpack(z, os.path.join(EXTRACT, os.path.basename(z).split(".")[0]))
    print("  done,", free())

# archives inside the extracted data: unpack, then delete the inner archive to save disk
for rnd in range(4):
    nested = [p for p in glob.glob(EXTRACT + "/**/*", recursive=True) if p.lower().endswith(ARCH)]
    if not nested: break
    print("round", rnd + 1, ":", len(nested), "nested archive(s)")
    for p in nested:
        if unpack(p, p + "_x"): os.remove(p)
    print("  ", free())

VIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv", ".mpg", ".mpeg", ".webm", ".m4v", ".3gp", ".wmv", ".flv")
files = [f for root in (EXTRACT, DATA_DIR) for f in glob.glob(root + "/**/*", recursive=True)
         if f.lower().endswith(VIDEO_EXT)]
files = sorted(set(files))
seen, uniq = set(), []
for f in files:                      # same video in two zips counts once
    k = (os.path.basename(os.path.dirname(f)).lower(), os.path.basename(f).lower(), os.path.getsize(f))
    if k not in seen: seen.add(k); uniq.append(f)
print(len(files) - len(uniq), "duplicate videos removed")
files = uniq
print(len(files), "videos found")
for f in files[:5]: print("  ", f)
MODE = "video"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
imgs = sorted(set(f for root in (EXTRACT, DATA_DIR) for f in glob.glob(root + "/**/*", recursive=True)
                  if f.lower().endswith(IMG_EXT)))
if imgs and files:
    # MIXED: word videos + letter/digit photos. Keep at most IMG_PER_LABEL photos per label so the
    # thousands of letter photos do not drown out the (much fewer) word videos.
    import random
    IMG_PER_LABEL = 40
    random.seed(0)
    by = collections.defaultdict(list)
    for f in imgs: by[os.path.basename(os.path.dirname(f))].append(f)
    kept = []
    for k in sorted(by):
        random.shuffle(by[k]); kept += sorted(by[k][:IMG_PER_LABEL])
    MODE = "mixed"
    print(f"MIXED mode: {len(files)} videos + {len(kept)} photos (max {IMG_PER_LABEL} per photo label, {len(by)} photo labels)")
    files = sorted(files) + kept
elif imgs:
    MODE, files = "image", imgs
    cnt = collections.Counter(os.path.basename(os.path.dirname(f)) for f in imgs)
    print(f"No videos, but {len(imgs)} IMAGES found -> image mode (one still hand pose per image).")
    print(len(cnt), "folders (= labels):", dict(sorted(cnt.items())))
if not files:
    # show what is actually there so the cause is obvious
    for root in (DATA_DIR, EXTRACT):
        allf = [p for p in glob.glob(root + "/**/*", recursive=True) if os.path.isfile(p)]
        ext = collections.Counter(os.path.splitext(p)[1].lower() or "(none)" for p in allf)
        print(root, "->", len(allf), "files, by extension:", dict(ext.most_common(10)))
        for p in allf[:5]: print("    ", p)
    raise FileNotFoundError(
        "No video files were found. Look at the extension counts above: if they show .rar/.7z, extract on your PC; "
        "if the folders are empty, the INCLUDE copy in Drive is incomplete (re-upload it); "
        "if INCLUDE is a shortcut, make a real copy of it in My Drive.")


## 2. Keypoint extraction
Each video becomes a `(30, 126)` array (2 hands x 21 points x xyz, each hand shifted to its wrist and scaled), plus the wrist positions for the 3D avatar. The label is the name of the folder the video sits in, with the leading number removed (for example `12. loud` becomes `loud`). Check the printout above: if the folder names are not the sign names, change `label_of`.\n\n**Image datasets:** if the dataset holds still photos (one folder per label, e.g. `isl_dataset/H/H_1.jpg`) instead of videos, the notebook switches to image mode automatically. Each photo gives one hand pose, which is repeated across the 30 frames, so the model learns held hand shapes (letters/digits), not moving signs. The folder name is used as the label as it is (lower-cased).

**Mixed mode:** if the INCLUDE folder holds word videos AND letter/digit photos, both are used (max 40 photos per label), so one model recognises letters, digits and words.

**Favouring words:** training weights the word classes more than the letters/digits (`WORD_FOCUS`), and the website's predictor multiplies word probabilities by `WORD_BOOST` (default 3) before ranking.

In [ ]:
import cv2, re, pickle, numpy as np, mediapipe as mp
from tqdm.auto import tqdm
import os, urllib.request

SEQ, FEATS = 30, 126

# Works with BOTH mediapipe APIs: the legacy mp.solutions (older versions)
# and the newer Tasks HandLandmarker (mediapipe >= 0.10.30, where solutions was removed).
if hasattr(mp, "solutions"):
    _hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
    def detect_hands(rgb):
        r = _hands.process(rgb)
        return [[(p.x, p.y, p.z) for p in h.landmark] for h in (r.multi_hand_landmarks or [])]
    print("mediapipe backend: legacy solutions")
else:
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
    MODEL = "/content/hand_landmarker.task"
    if not os.path.exists(MODEL):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task", MODEL)
    _lm = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL),
        running_mode=vision.RunningMode.IMAGE, num_hands=2, min_hand_detection_confidence=0.5))
    def detect_hands(rgb):
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        return [[(p.x, p.y, p.z) for p in h] for h in _lm.detect(img).hand_landmarks]
    print("mediapipe backend: Tasks HandLandmarker")

def hand_vec(rgb):
    hs = detect_hands(rgb)
    if not hs:
        return None
    out = np.zeros(FEATS, np.float32); wr = np.zeros(6, np.float32)
    for i, h in enumerate(sorted(hs, key=lambda h: h[0][0])[:2]):   # left-most hand first, same as the browser
        a = np.array(h, np.float32); wr[i * 3:(i + 1) * 3] = a[0]
        a -= a[0]; a /= max(np.abs(a).max(), 1e-6)
        out[i * 63:(i + 1) * 63] = a.flatten()
    return out, wr

def from_video(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n < 8:
        cap.release(); return None
    want = set(np.linspace(0, n - 1, min(n, 45)).astype(int).tolist()); last = max(want)
    vs, i = [], 0
    while i <= last:
        ok, f = cap.read()
        if not ok: break
        if i in want:
            if f.shape[1] > 640:
                f = cv2.resize(f, (640, int(f.shape[0] * 640 / f.shape[1])))
            v = hand_vec(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            if v is not None: vs.append(v)
        i += 1
    cap.release()
    if len(vs) < 8: return None
    sel = np.linspace(0, len(vs) - 1, SEQ).round().astype(int)
    return np.stack([v[0] for v in vs])[sel], np.stack([v[1] for v in vs])[sel]

def from_image(path):
    f = cv2.imread(path)
    if f is None: return None
    s = max(f.shape[:2])
    if s < 480 or s > 640:                                  # detector works best around 480-640 px
        k = 480 / s if s < 480 else 640 / s
        f = cv2.resize(f, (int(f.shape[1] * k), int(f.shape[0] * k)))
    return hand_vec(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))   # (126,), (6,) or None

def label_of(path):
    if path.lower().endswith(IMG_EXT):                      # photos: folder name IS the label (keeps "0", "1", ...)
        return os.path.basename(os.path.dirname(path)).strip().lower()
    return re.sub(r"^[\d\.\s_-]+", "", os.path.basename(os.path.dirname(path))).strip().lower()

assert len(files) > 0, "No videos in `files`: run the previous cells (section 1) first and fix the dataset path."
CKPT = f"{WORK}/features_ckpt_{MODE}.pkl"
read_one = lambda p: from_image(p) if p.lower().endswith(IMG_EXT) else from_video(p)
print("mode:", MODE, "|", len(files), "files")
done = {}
for m_ in ("image", "video", "mixed"):                       # reuse everything extracted in earlier runs
    p_ = f"{WORK}/features_ckpt_{m_}.pkl"
    if os.path.exists(p_): done.update(pickle.load(open(p_, "rb")))
print(len(done), "videos already processed")
for n, f in enumerate(tqdm(files)):
    if f in done: continue
    try:
        s = read_one(f)
    except Exception as e:
        s = None
    done[f] = None if s is None else (s[0], s[1], label_of(f))
    if n % 500 == 0:
        pickle.dump(done, open(CKPT, "wb"))
pickle.dump(done, open(CKPT, "wb"))

good = [done[f] for f in files if done.get(f) is not None]
def to_seq(v):                       # a photo gives (126,) -> repeat it over the 30 frames; a video is already (30,126)
    return np.repeat(v[None], SEQ, axis=0) if v.ndim == 1 else v
X = np.stack([to_seq(g[0]) for g in good]); W = np.stack([to_seq(g[1]) for g in good])
Y = np.array([g[2] for g in good])
print(f"{len(good)} usable videos, {len(set(Y))} labels, {len(files) - len(good)} skipped (no hands found)")
print("labels:", sorted(set(Y)))

## 3. Split and augmentation

In [ ]:
import torch, torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

keep = {k for k, v in Counter(Y).items() if v >= 4}
m = np.array([y in keep for y in Y]); X, W, Y = X[m], W[m], Y[m]
labels = sorted(set(Y)); l2i = {l: i for i, l in enumerate(labels)}
y = np.array([l2i[v] for v in Y])
idx = np.arange(len(y))
tr_i, tmp_i = train_test_split(idx, test_size=0.3, stratify=y, random_state=7)
va_i, te_i = train_test_split(tmp_i, test_size=0.5, stratify=y[tmp_i], random_state=7)
Xtr, ytr, Xva, yva, Xte, yte = X[tr_i], y[tr_i], X[va_i], y[va_i], X[te_i], y[te_i]
print(len(labels), "classes |", len(tr_i), "train |", len(va_i), "val |", len(te_i), "test")

class Signs(Dataset):
    def __init__(self, X, y, aug): self.X, self.y, self.aug = X, y, aug
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        a = self.X[i].copy()
        if self.aug:
            if np.random.rand() < 0.3:                                   # mirror (left-handed signers)
                b = a.reshape(SEQ, 2, 21, 3); b[..., 0] *= -1
                if b[:, 1].any(): b = b[:, ::-1]                         # two hands: swap slots
                a = np.ascontiguousarray(b).reshape(SEQ, FEATS)
            a = a + np.random.normal(0, 0.02, a.shape).astype(np.float32)       # jitter
            a = a * np.random.uniform(0.9, 1.1)                                  # scale
            a = a[np.sort(np.random.choice(SEQ, SEQ, replace=True))]            # speed change
        return torch.from_numpy(a.astype(np.float32)), int(self.y[i])

tl = DataLoader(Signs(Xtr, ytr, True), batch_size=32, shuffle=True)
vl = DataLoader(Signs(Xva, yva, False), batch_size=64)
tel = DataLoader(Signs(Xte, yte, False), batch_size=64)

## 4. Train (model code comes from the website's `model/isl_model.py`)

In [ ]:
import sys, glob, os, importlib
if "SITE" not in globals() or not os.path.isdir(SITE):
    hits = glob.glob("/content/site/**/model/isl_model.py", recursive=True)
    assert hits, "Website folder not found: run the section 1 cell first, then come back here."
    MODEL_DIR = os.path.dirname(hits[0]); SITE = os.path.dirname(MODEL_DIR)
if SITE not in sys.path: sys.path.insert(0, SITE)
importlib.invalidate_caches()
from model.isl_model import ISLNet
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
net = ISLNet(len(labels)).to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
EPOCHS, PATIENCE = 120, 25
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
# Favour WORD classes: inverse-frequency class weights (rare classes count more), and word classes get an extra factor.
WORD_FOCUS = 2.0
cnt = np.bincount(ytr, minlength=len(labels)).astype(np.float32)
wt = len(ytr) / (len(labels) * np.maximum(cnt, 1))
is_word = np.array([len(l) > 1 for l in labels])
wt[is_word] *= WORD_FOCUS
wt = torch.tensor(wt / wt.mean(), dtype=torch.float32).to(dev)
lossf = nn.CrossEntropyLoss(weight=wt, label_smoothing=0.1)
print(int(is_word.sum()), "word classes and", int((~is_word).sum()), "letter/digit classes")

def run_eval(model, loader, k=5):
    model.eval(); top1 = topk = n = 0
    with torch.no_grad():
        for xb, yb in loader:
            p = model(xb.to(dev)).cpu()
            top1 += (p.argmax(1) == yb).sum().item()
            topk += (p.topk(min(k, p.size(1)), 1).indices == yb[:, None]).any(1).sum().item(); n += len(yb)
    return top1 / n, topk / n

best, bad = 0, 0
BEST_PT = f"{WORK}/best.pt"      # in Drive, so it survives a Colab restart
for ep in range(EPOCHS):
    net.train()
    for xb, yb in tl:
        opt.zero_grad(); lossf(net(xb.to(dev)), yb.to(dev)).backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
    sched.step(); acc, _ = run_eval(net, vl)
    if acc > best: best, bad = acc, 0; torch.save(net.state_dict(), BEST_PT)
    else: bad += 1
    if ep % 5 == 0: print(f"epoch {ep:3d}  val acc {acc:.3f}  best {best:.3f}")
    if bad >= PATIENCE: print("early stop at epoch", ep); break
net.load_state_dict(torch.load(BEST_PT))
t1, t5 = run_eval(net, tel)
print(f"TEST  top-1 {t1:.3f}   top-5 {t5:.3f}")
# accuracy separately for words and for letters/digits
net.eval(); _P = []
with torch.no_grad():
    for xb, yb in tel: _P.append(net(xb.to(dev)).argmax(1).cpu())
_P = np.concatenate([p.numpy() for p in _P]); _w = is_word[yte]
if _w.any():  print(f"TEST  words {(_P[_w] == yte[_w]).mean():.3f}  ({int(_w.sum())} samples)")
if (~_w).any(): print(f"TEST  letters/digits {(_P[~_w] == yte[~_w]).mean():.3f}  ({int((~_w).sum())} samples)")

## 4b. Federated update simulation
Each simulated client (think: one hospital desk) trains on its own share of the hand-point data and only the weights are averaged (FedAvg). No video, and no raw data, leaves a client. The averaged model is kept only if it is about as good as the centrally trained one.

In [ ]:
def local_train(sd, Xc, yc, epochs=2):
    m = ISLNet(len(labels)).to(dev); m.load_state_dict(sd); m.train()
    o = torch.optim.AdamW(m.parameters(), lr=3e-4)
    dl = DataLoader(Signs(Xc, yc, True), batch_size=16, shuffle=True)
    for _ in range(epochs):
        for xb, yb in dl:
            o.zero_grad(); lossf(m(xb.to(dev)), yb.to(dev)).backward(); o.step()
    return {k: v.cpu() for k, v in m.state_dict().items()}, len(Xc)

def fedavg(results):
    total = sum(n for _, n in results)
    return {k: sum(sd[k].float() * n for sd, n in results) / total for k in results[0][0]}

CLIENTS, ROUNDS = 5, 5
parts = np.array_split(np.random.permutation(len(Xtr)), CLIENTS)
g = {k: v.cpu() for k, v in net.state_dict().items()}
for rnd in range(ROUNDS):
    g = fedavg([local_train(g, Xtr[p], ytr[p]) for p in parts])
    net.load_state_dict(g)
    print("round", rnd + 1, "val acc", round(run_eval(net, vl)[0], 3))
if run_eval(net, vl)[0] >= best - 0.01:
    torch.save(net.state_dict(), BEST_PT); print("federated model kept")
else:
    net.load_state_dict(torch.load(BEST_PT)); print("centrally trained model kept (it was better)")
print("TEST  top-1 %.3f   top-5 %.3f" % run_eval(net, tel))

In [ ]:
# Optional: the same local step packaged as a Flower client (pip install flwr to use it)
try:
    import flwr as fl
    class LandmarkClient(fl.client.NumPyClient):
        def __init__(self, Xc, yc): self.X, self.y = Xc, yc
        def get_parameters(self, config): return [v.cpu().numpy() for v in net.state_dict().values()]
        def fit(self, params, config):
            sd = {k: torch.tensor(v) for k, v in zip(net.state_dict().keys(), params)}
            new, n = local_train(sd, self.X, self.y)
            return [v.numpy() for v in new.values()], n, {}
        def evaluate(self, params, config): return 0.0, len(self.X), {}
except ImportError:
    print("flwr not installed, skipping the Flower wrapper")

## 5. Save into the website folder

In [ ]:
import sys, glob, os, importlib
if "SITE" not in globals() or not os.path.isdir(SITE):
    hits = glob.glob("/content/site/**/model/isl_model.py", recursive=True)
    assert hits, "Website folder not found: run the section 1 cell first, then come back here."
    MODEL_DIR = os.path.dirname(hits[0]); SITE = os.path.dirname(MODEL_DIR)
if SITE not in sys.path: sys.path.insert(0, SITE)
importlib.invalidate_caches()
import json
if "net" not in globals():            # runtime was restarted: rebuild the model from the saved best weights
    from model.isl_model import ISLNet
    assert os.path.exists(BEST_PT), "No trained weights found: run the section 4 training cell first."
    net = ISLNet(len(labels)); net.load_state_dict(torch.load(BEST_PT, map_location="cpu"))
net.cpu().eval()
torch.save(net.state_dict(), f"{MODEL_DIR}/isl_model.pt")
json.dump(labels, open(f"{MODEL_DIR}/labels.json", "w"), ensure_ascii=False)

# pose library for the 3D avatar: the real sample closest to each sign's average
pose = {}
for i, lab in enumerate(labels):
    S, Wi = X[y == i], W[y == i]; mean = S.mean(0)
    k = np.argmin(((S - mean) ** 2).sum((1, 2)))
    pose[lab] = {"f": np.round(S[k], 3).tolist(), "w": np.round(Wi[k], 3).tolist()}
json.dump(pose, open(f"{MODEL_DIR}/pose_library.json", "w"))

# small replay buffer for continual learning
rx, ry = [], []
for i in range(len(labels)):
    S = Xtr[ytr == i][:5]; rx += list(S); ry += [i] * len(S)
torch.save({"x": torch.tensor(np.stack(rx)), "y": torch.tensor(ry)}, f"{MODEL_DIR}/replay.pt")

# check that the website's own predictor loads the files and answers
from model.predictor import Predictor
P = Predictor(MODEL_DIR)
print("predictor ready:", P.ready)
print("true:", labels[yte[0]], "| predicted:", P.predict(Xte[0]))

## 6. Zip the finished website

In [ ]:
OUT = f"{DRIVE}/ISL_Bridge_trained.zip"
root = os.path.dirname(SITE)
if os.path.exists(OUT): os.remove(OUT)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for dp, _, fs in os.walk(SITE):
        if "__pycache__" in dp: continue
        for f in fs:
            p = os.path.join(dp, f); z.write(p, os.path.relpath(p, root))
print("saved", OUT, round(os.path.getsize(OUT) / 1e6, 1), "MB")
from google.colab import files
files.download(OUT)